# RecordDiff: Recorded-Law Fidelity

## 1 · Setup

In [ ]:
import os, time, numpy as np, torch
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)
if device == "cpu":
    print("WARNING: no GPU — generation and the sequence C2ST will be slow.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
# cohort path
V2_CACHE = "..."
# model checkpoint path
CKPT     = "..."

MAX_SYNTH   = 5000
C2ST_SPLITS = 5
N_PERM      = 80
MMD_MAX_N   = 1500
SEQ_EPOCHS  = 8
SEQ_HIDDEN  = 64
WINS        = (0.005, 0.995)
SEED        = 0
rng = np.random.default_rng(SEED)
print("MAX_SYNTH", MAX_SYNTH, "| C2ST splits", C2ST_SPLITS, "| seq epochs", SEQ_EPOCHS)

## 3 · Model core

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# utils
def timestep_embedding(t, dim):
    """Sinusoidal embedding of diffusion step t:(B,) -> (B,dim)."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / max(half, 1))
    args = t.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        emb = F.pad(emb, (0, 1))
    return emb


def causal_hist(m):
    """m:(B,T,V) {0,1} -> (B,T,2V): [count-before-t / T, steps-since-last-obs-before-t / T].
    Strictly causal (uses only t' < t), matching the incremental generation-time update."""
    B, T, V = m.shape
    csum = torch.cumsum(m, dim=1)
    count_excl = csum - m
    idx = torch.arange(T, device=m.device).view(1, T, 1).float().expand(B, T, V)
    obs_pos = torch.where(m > 0.5, idx, torch.full_like(m, -1.0))
    shifted = torch.cat([torch.full((B, 1, V), -1.0, device=m.device), obs_pos[:, :-1, :]], dim=1).contiguous()
    last_obs_excl = torch.cummax(shifted, dim=1).values
    dt = idx - last_obs_excl
    return torch.cat([count_excl / T, dt / T], dim=-1)


class DiffusionSchedule:
    """Standard DDPM linear-beta schedule with precomputed coefficients."""
    def __init__(self, n_steps=100, beta_start=1e-4, beta_end=2e-2):
        betas = torch.linspace(beta_start, beta_end, n_steps)
        alphas = 1.0 - betas
        abar = torch.cumprod(alphas, dim=0)
        abar_prev = torch.cat([torch.ones(1), abar[:-1]])
        self.n_steps = n_steps
        self.betas = betas
        self.alphas = alphas
        self.abar = abar
        self.sqrt_abar = torch.sqrt(abar)
        self.sqrt_one_minus_abar = torch.sqrt(1.0 - abar)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        self.posterior_var = betas * (1.0 - abar_prev) / (1.0 - abar)

    def to(self, device):
        for k, v in list(self.__dict__.items()):
            if torch.is_tensor(v):
                setattr(self, k, v.to(device))
        return self

    def q_sample(self, x0, t, noise):
        sa = self.sqrt_abar[t].view(-1, 1, 1)
        soma = self.sqrt_one_minus_abar[t].view(-1, 1, 1)
        return sa * x0 + soma * noise


class Denoiser(nn.Module):
    """Predicts diffusion noise per timestep: eps_theta(x_noisy, cond, t). Applied vectorised over T."""
    def __init__(self, V, d_cond, d_temb=64, hidden=256):
        super().__init__()
        self.d_temb = d_temb
        self.net = nn.Sequential(
            nn.Linear(V + d_cond + d_temb, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, V),
        )

    def forward(self, x, cond, t):
        temb = timestep_embedding(t, self.d_temb)
        temb = temb[:, None, :].expand(-1, x.shape[1], -1)
        return self.net(torch.cat([x, cond, temb], dim=-1))


#  model
class RecordDiff(nn.Module):
    def __init__(self, V, d_c=8, d_z=32, d_e=128, d_rnn=128, d_comb=128,
                 hidden_mask=256, hidden_den=256, d_temb=64, n_diff=100, d_u=8):
        super().__init__()
        self.V, self.d_c, self.d_z, self.d_u = V, d_c, d_z, d_u
        self.diff = DiffusionSchedule(n_diff)

        self.emb = nn.Sequential(nn.Linear(2 * V, d_e), nn.SiLU(), nn.Linear(d_e, d_e))
        self.bigru = nn.GRU(d_e, d_rnn, batch_first=True, bidirectional=True)
        self.comb_z = nn.Linear(d_z, d_comb)
        self.comb_g = nn.Linear(2 * d_rnn, d_comb)
        self.q_mu = nn.Linear(d_comb, d_z)
        self.q_ls = nn.Linear(d_comb, d_z)
        if d_u > 0:
            self.u_mu = nn.Linear(2 * d_rnn, d_u)
            self.u_ls = nn.Linear(2 * d_rnn, d_u)
        self.p0_mu = nn.Linear(d_c, d_z)
        self.p0_ls = nn.Linear(d_c, d_z)
        self.tr_body = nn.Sequential(nn.Linear(d_z + d_c, d_comb), nn.SiLU())
        self.tr_mu = nn.Linear(d_comb, d_z)
        self.tr_ls = nn.Linear(d_comb, d_z)
        self.mask_head = nn.Sequential(
            nn.Linear(d_z + d_c + 2 * V + d_u, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, hidden_mask), nn.SiLU(),
            nn.Linear(hidden_mask, V))
        self.d_cond = d_z + d_c + V + 2 * V
        self.denoiser = Denoiser(V, self.d_cond, d_temb, hidden_den)
        self.register_buffer("norm_mean", torch.zeros(V))
        self.register_buffer("norm_std", torch.ones(V))

    # parameter groups for the 3-phase schedule
    def params_encoder(self):
        mods = [self.emb, self.bigru, self.comb_z, self.comb_g, self.q_mu, self.q_ls,
                self.p0_mu, self.p0_ls, self.tr_body, self.tr_mu, self.tr_ls]
        if self.d_u > 0:
            mods += [self.u_mu, self.u_ls]
        return [p for mod in mods for p in mod.parameters()]

    def params_value(self):
        return list(self.denoiser.parameters())

    def params_mask(self):
        return list(self.mask_head.parameters())

    def set_normalizer(self, mean, std):
        self.norm_mean.data = mean.to(self.norm_mean.device)
        self.norm_std.data = std.clamp(min=1e-3).to(self.norm_std.device)

    def _prior_step(self, z_prev, c, t):
        if t == 0:
            return self.p0_mu(c), self.p0_ls(c)
        body = self.tr_body(torch.cat([z_prev, c], dim=-1))
        return z_prev + self.tr_mu(body), self.tr_ls(body)

    def infer(self, y, m, c):
        """Amortised posterior over z_{1:T} and the patient random effect u.
        Returns Z, KL_z, u, KL_u (all latent KLs against their priors)."""
        B, T, V = y.shape
        e = self.emb(torch.cat([y, m], dim=-1))
        g, _ = self.bigru(e)
        if self.d_u > 0:
            gpool = g.mean(1)
            mu_u, ls_u = self.u_mu(gpool), self.u_ls(gpool)
            std_u = F.softplus(ls_u) + 1e-4
            u = mu_u + std_u * torch.randn_like(std_u)
            kl_u = (-torch.log(std_u) + 0.5 * (std_u ** 2 + mu_u ** 2) - 0.5)
        else:
            u = torch.zeros(B, 0, device=y.device)
            kl_u = torch.zeros(B, 0, device=y.device)
        z_prev = torch.zeros(B, self.d_z, device=y.device)
        Zs, KLs = [], []
        for t in range(T):
            hc = 0.5 * (torch.tanh(self.comb_z(z_prev)) + self.comb_g(g[:, t, :]))
            mu_q, ls_q = self.q_mu(hc), self.q_ls(hc)
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            std_q = F.softplus(ls_q) + 1e-4
            std_p = F.softplus(ls_p) + 1e-4
            z = mu_q + std_q * torch.randn_like(std_q)
            kl = (torch.log(std_p / std_q)
                  + (std_q ** 2 + (mu_q - mu_p) ** 2) / (2 * std_p ** 2) - 0.5)
            Zs.append(z); KLs.append(kl)
            z_prev = z
        Z = torch.stack(Zs, dim=1)
        KL = torch.stack(KLs, dim=1)
        return Z, KL, u, kl_u

    def losses(self, y, m, c, free_bits=0.02):
        """y assumed already normalised; m in {0,1}; c:(B,d_c). Returns loss dict."""
        B, T, V = y.shape
        hist = causal_hist(m)
        Z, KL, u, kl_u = self.infer(y, m, c)
        c_seq = c[:, None, :].expand(-1, T, -1)
        u_seq = u[:, None, :].expand(-1, T, -1)

        mask_logits = self.mask_head(torch.cat([Z, c_seq, hist, u_seq], dim=-1))
        L_mask = F.binary_cross_entropy_with_logits(mask_logits, m)

        x0 = y
        tau = torch.randint(0, self.diff.n_steps, (B,), device=y.device)
        noise = torch.randn_like(x0)
        x_noisy = self.diff.q_sample(x0, tau, noise)
        cond = torch.cat([Z, c_seq, m, hist], dim=-1)
        eps_pred = self.denoiser(x_noisy, cond, tau)
        se = (eps_pred - noise) ** 2
        L_value = (se * m).sum() / m.sum().clamp(min=1.0)

        KL_fb = (torch.clamp(KL, min=free_bits).sum(-1).mean()
                 + torch.clamp(kl_u, min=free_bits).sum(-1).mean())
        return {"L_value": L_value, "L_mask": L_mask, "KL": KL_fb}

    # generation
    @torch.no_grad()
    def _ddpm_sample(self, cond):
        """Reverse DDPM for a single timestep. cond:(n,d_cond) -> x:(n,V) (normalised)."""
        n = cond.shape[0]
        dev = cond.device
        x = torch.randn(n, 1, self.V, device=dev)
        cond1 = cond[:, None, :]
        for i in reversed(range(self.diff.n_steps)):
            tau = torch.full((n,), i, dtype=torch.long, device=dev)
            eps = self.denoiser(x, cond1, tau)
            mean = self.diff.sqrt_recip_alphas[i] * (
                x - self.diff.betas[i] / self.diff.sqrt_one_minus_abar[i] * eps)
            if i > 0:
                x = mean + torch.sqrt(self.diff.posterior_var[i]) * torch.randn_like(x)
            else:
                x = mean
        return x[:, 0, :]

    @torch.no_grad()
    def generate(self, n, c, T, denorm=True, return_x=False):
        """Ancestral sample of recorded reality. Returns (Y, M) [+ complete X if return_x],
        denormalised if denorm=True. X is the pre-mask complete value tensor."""
        dev = c.device
        V = self.V
        count_run = torch.zeros(n, V, device=dev)
        last_obs = -torch.ones(n, V, device=dev)
        z_prev = torch.zeros(n, self.d_z, device=dev)
        u = torch.randn(n, self.d_u, device=dev) if self.d_u > 0 else torch.zeros(n, 0, device=dev)
        M = torch.zeros(n, T, V, device=dev)
        Y = torch.zeros(n, T, V, device=dev)
        X = torch.zeros(n, T, V, device=dev) if return_x else None
        for t in range(T):
            mu_p, ls_p = self._prior_step(z_prev, c, t)
            z = mu_p + (F.softplus(ls_p) + 1e-4) * torch.randn_like(mu_p)
            hist = torch.cat([count_run / T, (t - last_obs) / T], dim=-1)
            m_t = torch.bernoulli(torch.sigmoid(self.mask_head(torch.cat([z, c, hist, u], dim=-1))))
            x_t = self._ddpm_sample(torch.cat([z, c, m_t, hist], dim=-1))
            if denorm:
                x_t = x_t * self.norm_std + self.norm_mean
            M[:, t, :] = m_t
            Y[:, t, :] = m_t * x_t
            if return_x:
                X[:, t, :] = x_t
            last_obs = torch.where(m_t > 0.5, torch.full_like(last_obs, float(t)), last_obs)
            count_run = count_run + m_t
            z_prev = z
        return (Y, M, X) if return_x else (Y, M)


# data helpers
def fit_normalizer(y, m):
    """Per-variable mean/std over observed (m==1) entries. y,m:(N,T,V) tensors."""
    V = y.shape[-1]
    mean = torch.zeros(V); std = torch.ones(V)
    for v in range(V):
        vals = y[:, :, v][m[:, :, v] > 0.5]
        if vals.numel() > 10:
            mean[v] = vals.mean()
            std[v] = vals.std().clamp(min=1e-3)
    return mean, std


def normalize(y, m, mean, std):
    yn = (y - mean) / std
    return yn * m


def set_requires_grad(params, flag):
    for p in params:
        p.requires_grad_(flag)


def train_recorddiff(model, y, m, c, epochs=(8, 4, 8), batch=256, lr=1e-3,
                     beta_max=1.0, lambda_m=1.0, free_bits=0.02, val_frac=0.1,
                     clip=5.0, device="cpu", seed=0, verbose=True):
    """3-phase amortised-VI training.
        Phase 1 (representation): encoder + value diffusion + prior,  loss = L_value + beta*KL
        Phase 2 (policy):         freeze the above, train mask head,   loss = L_mask
        Phase 3 (joint):          everything,  loss = L_value + lambda_m*L_mask + beta_max*KL
    y is RAW (normalised internally via model.norm_*). Returns loss history."""
    torch.manual_seed(seed)
    model.to(device); model.diff.to(device)
    N = y.shape[0]
    perm = torch.randperm(N)
    n_val = int(N * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    e1, e2, e3 = epochs
    total = e1 + e2 + e3
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    mean, std = model.norm_mean.cpu(), model.norm_std.cpu()
    hist = {"phase": [], "L_value": [], "L_mask": [], "KL": [], "val_total": []}

    def run_batches(idx, train=True):
        agg = {"L_value": 0.0, "L_mask": 0.0, "KL": 0.0, "tot": 0.0, "nb": 0}
        order = idx[torch.randperm(len(idx))] if train else idx
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]
            yb = normalize(y[bi], m[bi], mean, std).to(device)
            mb = m[bi].to(device)
            cb = c[bi].to(device)
            out = model.losses(yb, mb, cb, free_bits=free_bits)
            if phase == 1:
                loss = out["L_value"] + beta * out["KL"]
            elif phase == 2:
                loss = out["L_mask"]
            else:
                loss = out["L_value"] + lambda_m * out["L_mask"] + beta_max * out["KL"]
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip); opt.step()
            for k in ("L_value", "L_mask", "KL"):
                agg[k] += float(out[k].detach())
            agg["tot"] += float(loss.detach()); agg["nb"] += 1
        for k in ("L_value", "L_mask", "KL", "tot"):
            agg[k] /= max(agg["nb"], 1)
        return agg

    for ep in range(total):
        if ep < e1:
            phase = 1; beta = beta_max * min(1.0, (ep + 1) / max(e1, 1))
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), False)
        elif ep < e1 + e2:
            phase = 2; beta = beta_max
            set_requires_grad(model.params_encoder(), False)
            set_requires_grad(model.params_value(), False)
            set_requires_grad(model.params_mask(), True)
        else:
            phase = 3; beta = beta_max
            set_requires_grad(model.params_encoder(), True)
            set_requires_grad(model.params_value(), True)
            set_requires_grad(model.params_mask(), True)

        model.train(); tr = run_batches(tr_idx, train=True)
        model.eval()
        with torch.no_grad():
            va = run_batches(val_idx, train=False) if n_val > 0 else tr
        hist["phase"].append(phase)
        hist["L_value"].append(tr["L_value"]); hist["L_mask"].append(tr["L_mask"])
        hist["KL"].append(tr["KL"]); hist["val_total"].append(va["tot"])
        if verbose:
            print(f"ep {ep:3d} | phase {phase} | "
                  f"L_value {tr['L_value']:.4f}  L_mask {tr['L_mask']:.4f}  KL {tr['KL']:.4f} "
                  f"| val_total {va['tot']:.4f}")
    return hist

# synthetic
def make_synthetic(n=512, T=12, V=6, seed=0, device="cpu"):
    """Toy MNAR data: latent severity drives both values and (informatively) the mask."""
    g = torch.Generator().manual_seed(seed)
    sev = torch.rand(n, 1, 1, generator=g)
    base = torch.randn(n, 1, V, generator=g)
    drift = torch.linspace(0, 1, T).view(1, T, 1) * (sev - 0.5) * 4.0
    x = base + drift + 0.3 * torch.randn(n, T, V, generator=g)
    logit = -0.5 + 2.0 * sev + 0.6 * (x > 1.0).float() + 0.4 * torch.randn(n, T, V, generator=g)
    m = torch.bernoulli(torch.sigmoid(logit), generator=g)
    y = x * m
    c = torch.cat([sev.view(n, 1), torch.randn(n, 7, generator=g)], dim=-1)
    return y.to(device), m.to(device), c.to(device)


## 4 · Fidelity helpers

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.stats import ks_2samp

# featurizers
def mask_features(m):
    """m:(N,T,V) {0,1} -> (N,4V): count, rate, first-obs-time, last-obs-time."""
    N, T, V = m.shape
    count = m.sum(1).astype(np.float32)
    rate = count / T
    obs = m > 0.5
    first = np.where(obs.any(1), np.argmax(obs, 1), T).astype(np.float32) / T
    rev = obs[:, ::-1, :]
    last_idx = np.where(obs.any(1), (T - 1 - np.argmax(rev, 1)), -1.0).astype(np.float32)
    last = (last_idx + 1.0) / T
    return np.concatenate([count, rate, first, last], axis=1).astype(np.float32)


def value_features(y, m):
    """Per-variable summary over OBSERVED entries: mean, std, min, max, last value."""
    N, T, V = y.shape
    obs = m > 0.5
    cnt = np.maximum(m.sum(1), 1)
    mean = (y.sum(1) / cnt).astype(np.float32)
    var = np.maximum((y * y).sum(1) / cnt - mean ** 2, 0.0)
    std = np.sqrt(var).astype(np.float32)
    any_obs = obs.any(1)
    vmin = np.where(any_obs, np.where(obs, y, np.inf).min(1), 0.0).astype(np.float32)
    vmax = np.where(any_obs, np.where(obs, y, -np.inf).max(1), 0.0).astype(np.float32)
    rev = obs[:, ::-1, :]
    last_t = np.where(any_obs, T - 1 - np.argmax(rev, 1), 0)
    last = np.take_along_axis(y, last_t[:, None, :], axis=1)[:, 0, :]
    last = np.where(any_obs, last, 0.0).astype(np.float32)
    feats = np.concatenate([mean, std, vmin, vmax, last], axis=1)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


def summary_features(m, y, c=None, channel="joint"):
    """Assemble discriminator features. channel in {mask, value, joint}."""
    parts = []
    if channel in ("mask", "joint"):
        parts.append(mask_features(m))
    if channel in ("value", "joint"):
        parts.append(value_features(y, m))
    X = np.concatenate(parts, axis=1)
    if c is not None:
        X = np.concatenate([X, np.asarray(c, np.float32)], axis=1)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


# MMD (RBF, unbiased, median-heuristic)
def _sqdist(A, B):
    a2 = (A * A).sum(1)[:, None]
    b2 = (B * B).sum(1)[None, :]
    return np.maximum(a2 + b2 - 2.0 * A @ B.T, 0.0)


def mmd2_rbf(X, Y, max_n=1500, seed=0):
    """Unbiased RBF-MMD^2 between X and Y (subsampled to max_n each)."""
    rng = np.random.default_rng(seed)
    if len(X) > max_n:
        X = X[rng.choice(len(X), max_n, replace=False)]
    if len(Y) > max_n:
        Y = Y[rng.choice(len(Y), max_n, replace=False)]
    sc = StandardScaler().fit(np.vstack([X, Y]))
    X, Y = sc.transform(X), sc.transform(Y)
    Kxx, Kyy, Kxy = _sqdist(X, X), _sqdist(Y, Y), _sqdist(X, Y)
    med = np.median(np.concatenate([Kxx[np.triu_indices_from(Kxx, 1)], Kxy.ravel()]))
    sigma2 = med if med > 1e-9 else 1.0
    kxx = np.exp(-Kxx / sigma2); kyy = np.exp(-Kyy / sigma2); kxy = np.exp(-Kxy / sigma2)
    n, m = len(X), len(Y)
    np.fill_diagonal(kxx, 0.0); np.fill_diagonal(kyy, 0.0)
    return float(kxx.sum() / (n * (n - 1)) + kyy.sum() / (m * (m - 1)) - 2.0 * kxy.mean())


# C2ST (classifier two-sample test)
def _balance(Xr, Xs, seed):
    rng = np.random.default_rng(seed)
    n = min(len(Xr), len(Xs))
    Xr = Xr[rng.choice(len(Xr), n, replace=False)]
    Xs = Xs[rng.choice(len(Xs), n, replace=False)]
    X = np.vstack([Xr, Xs]); lab = np.r_[np.ones(n), np.zeros(n)].astype(int)
    return X, lab


def _clf():
    return HistGradientBoostingClassifier(max_depth=4, max_iter=150, learning_rate=0.1)


def c2st(Xr, Xs, n_splits=5, test_frac=0.3, seed=0):
    """Real(1) vs synthetic(0) discriminator. Returns dict with acc/auroc mean+CI over splits.
    Under perfect fidelity ~0.50."""
    X, lab = _balance(Xr, Xs, seed)
    accs, aucs = [], []
    sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_frac, random_state=seed)
    for tr, te in sss.split(X, lab):
        sc = StandardScaler().fit(X[tr])
        clf = _clf().fit(sc.transform(X[tr]), lab[tr])
        p = clf.predict_proba(sc.transform(X[te]))[:, 1]
        accs.append(((p > 0.5).astype(int) == lab[te]).mean())
        aucs.append(roc_auc_score(lab[te], p))
    accs, aucs = np.array(accs), np.array(aucs)
    return dict(acc=float(accs.mean()), acc_lo=float(np.percentile(accs, 2.5)),
                acc_hi=float(np.percentile(accs, 97.5)),
                auc=float(aucs.mean()), auc_lo=float(np.percentile(aucs, 2.5)),
                auc_hi=float(np.percentile(aucs, 97.5)))


def c2st_permutation_pvalue(Xr, Xs, n_perm=100, seed=0):
    """p-value that the C2ST accuracy exceeds the label-shuffle null."""
    X, lab = _balance(Xr, Xs, seed)
    def one(labels, sd):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=sd)
        tr, te = next(sss.split(X, labels))
        sc = StandardScaler().fit(X[tr])
        clf = _clf().fit(sc.transform(X[tr]), labels[tr])
        return (((clf.predict_proba(sc.transform(X[te]))[:, 1] > 0.5).astype(int)) == labels[te]).mean()
    obs = one(lab, seed)
    rng = np.random.default_rng(seed)
    null = np.array([one(rng.permutation(lab), seed + 1 + i) for i in range(n_perm)])
    return float(obs), float(((null >= obs).sum() + 1) / (n_perm + 1))


# distributional diagnostics
def ks_per_variable(yr, mr, ys, ms):
    V = yr.shape[-1]; out = np.zeros(V)
    for v in range(V):
        a = yr[:, :, v][mr[:, :, v] > 0.5]; b = ys[:, :, v][ms[:, :, v] > 0.5]
        out[v] = ks_2samp(a, b).statistic if (a.size > 5 and b.size > 5) else np.nan
    return out


def obs_per_record_ks(mr, ms):
    return float(ks_2samp(mr.sum((1, 2)), ms.sum((1, 2))).statistic)


def cooccurrence(m):
    """(V,V) mean co-measurement rate across patient-hours."""
    N, T, V = m.shape
    flat = m.reshape(-1, V).astype(np.float32)
    return (flat.T @ flat) / flat.shape[0]


def value_corr_nan(y, m):
    import pandas as pd
    N, T, V = y.shape
    arr = y.reshape(-1, V).astype(float).copy()
    arr[m.reshape(-1, V) < 0.5] = np.nan
    return pd.DataFrame(arr).corr().values


def frob(A, B):
    A = np.nan_to_num(A); B = np.nan_to_num(B)
    return float(np.sqrt(((A - B) ** 2).sum()))


## 5 · Sequence C2ST


In [ ]:
import torch.nn as nn

class SeqDisc(nn.Module):
    def __init__(self, V, hidden=64):
        super().__init__()
        self.gru = nn.GRU(2 * V, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x):
        o, _ = self.gru(x)
        return self.head(o.mean(1)).squeeze(-1)

def seq_input(m, y, mu, sd):
    yz = ((y - mu[None, None, :]) / sd[None, None, :]) * m
    return np.concatenate([m, np.nan_to_num(yz)], axis=-1).astype(np.float32)

def seq_c2st(Ir, Is, epochs=8, hidden=64, batch=256, seed=0):
    torch.manual_seed(seed)
    n = min(len(Ir), len(Is))
    g = np.random.default_rng(seed)
    Ir = Ir[g.choice(len(Ir), n, replace=False)]; Is = Is[g.choice(len(Is), n, replace=False)]
    X = np.concatenate([Ir, Is], 0); lab = np.r_[np.ones(n), np.zeros(n)].astype(np.float32)
    perm = g.permutation(2 * n); X, lab = X[perm], lab[perm]
    ntr = int(1.6 * n)
    Xtr, ytr, Xte, yte = X[:ntr], lab[:ntr], X[ntr:], lab[ntr:]
    V = Ir.shape[-1] // 2
    net = SeqDisc(V, hidden).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    Xtr_t = torch.from_numpy(Xtr).to(device); ytr_t = torch.from_numpy(ytr).to(device)
    for ep in range(epochs):
        net.train(); order = torch.randperm(len(Xtr_t))
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(net(Xtr_t[bi]), ytr_t[bi])
            loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        p = torch.sigmoid(net(torch.from_numpy(Xte).to(device))).cpu().numpy()
    from sklearn.metrics import roc_auc_score
    acc = ((p > 0.5).astype(int) == yte.astype(int)).mean()
    auc = roc_auc_score(yte, p) if len(np.unique(yte)) > 1 else float("nan")
    return float(acc), float(auc)
print("sequence C2ST defined")

## 6 · Load checkpoint, generate synthetic cohort, build ablations

In [ ]:
d = np.load(V2_CACHE, allow_pickle=True)
m_real_all = d["m"].astype(np.float32); y_real_all = d["y"].astype(np.float32)
VAR_NAMES = list(d["var_names"]); VAR_CLASS = [str(x) for x in d["var_class"]]
c_all = d["c"].astype(np.float32) if "c" in d.files else None
N_all, T, V = m_real_all.shape

ckpt = torch.load(CKPT, map_location=device, weights_only=False)
cfg = ckpt["config"]; D_C = cfg["d_c"]
model = RecordDiff(**cfg).to(device); model.load_state_dict(ckpt["state_dict"])
model.diff.to(device); model.eval()
print(f"loaded checkpoint | d_c={D_C} d_z={cfg['d_z']} n_diff={cfg['n_diff']}")

COV_OK = (c_all is not None) and (c_all.shape[1] == D_C) and (D_C > 0)
NG = min(MAX_SYNTH, N_all)
gi = rng.choice(N_all, NG, replace=False)
if COV_OK:
    c_gen = torch.from_numpy(c_all[gi]).to(device); c_feat = c_all[gi]
    print("covariates: using real covariates in the discriminator")
else:
    c_gen = torch.zeros(NG, D_C, device=device); c_feat = None
    print("covariates: checkpoint trained without covariates -> excluded from discriminator")

t0 = time.time()
with torch.no_grad():
    Y_s, M_s, X_s = model.generate(NG, c_gen, T, return_x=True)
Y_s, M_s, X_s = Y_s.cpu().numpy(), M_s.cpu().numpy(), X_s.cpu().numpy()
print(f"generated {NG} synthetic records in {time.time()-t0:.0f}s")

DISCRETE = {"GCS_eye": (1, 4), "GCS_verbal": (1, 5), "GCS_motor": (1, 6)}
for v in range(V):
    obs = y_real_all[:, :, v][m_real_all[:, :, v] > 0.5]
    if obs.size > 200:
        lo, hi = np.quantile(obs, WINS); X_s[:, :, v] = np.clip(X_s[:, :, v], lo, hi)
for _name, (lo, hi) in DISCRETE.items():
    if _name in VAR_NAMES:
        vi = VAR_NAMES.index(_name); X_s[:, :, vi] = np.clip(np.round(X_s[:, :, vi]), lo, hi)
Y_s = (M_s * X_s).astype(np.float32)
print("Tier-0 hygiene: synthetic values clipped to real support; GCS rounded to integers")

ri = rng.choice(N_all, NG, replace=False)
m_real, y_real = m_real_all[ri].copy(), y_real_all[ri].copy()
for v in range(V):
    vals = y_real_all[:, :, v][m_real_all[:, :, v] > 0.5]
    if vals.size > 200:
        lo, hi = np.quantile(vals, WINS); y_real[:, :, v] = np.clip(y_real[:, :, v], lo, hi) * m_real[:, :, v]
c_real = c_all[ri] if COV_OK else None

real_rate = m_real_all.mean((0, 1))
M_indep = (rng.random((NG, T, V)) < real_rate[None, None, :]).astype(np.float32)
Y_indep = (M_indep * X_s).astype(np.float32)
rj = rng.choice(N_all, NG, replace=(NG > N_all))
M_replay = m_real_all[rj]; Y_replay = (M_replay * X_s).astype(np.float32)
print("cohorts ready: real | RecordDiff | independent-mask | mask-replay")
print(f"mask density  real {m_real.mean():.3f} | synth {M_s.mean():.3f} | indep {M_indep.mean():.3f}")

## 7 · Smoke test

In [ ]:
ys_r, ms_r, cs_r = make_synthetic(n=200, T=10, V=6, seed=1)
sm = RecordDiff(V=6, d_c=8, d_z=8, d_e=32, d_rnn=32, d_comb=32,
                hidden_mask=64, hidden_den=64, d_temb=32, n_diff=15).to(device)
sm.diff.to(device); sm.set_normalizer(*fit_normalizer(ys_r, ms_r))
with torch.no_grad():
    Yts, Mts, _ = sm.generate(200, torch.zeros(200, 8, device=device), T=10, return_x=True)
mr_, yr_ = ms_r.numpy(), ys_r.numpy(); ms_, ys_ = Mts.cpu().numpy(), Yts.cpu().numpy()

a_sum = c2st(summary_features(mr_, yr_), summary_features(ms_, ys_), n_splits=3)["acc"]
mmd_v = mmd2_rbf(summary_features(mr_, yr_), summary_features(ms_, ys_), max_n=200)
mu_, sd_ = value_features(yr_, mr_)[:, :6].mean(0) * 0 + yr_[mr_ > 0.5].mean(), \
           np.ones(6) * max(yr_[mr_ > 0.5].std(), 1e-3)
a_seq, _ = seq_c2st(seq_input(mr_, yr_, mu_, sd_), seq_input(ms_, ys_, mu_, sd_), epochs=2, hidden=16)
assert 0.0 <= a_sum <= 1.0 and 0.0 <= a_seq <= 1.0 and np.isfinite(mmd_v), "smoke: bad output"
print(f"SMOKE OK — summary C2ST {a_sum:.3f} | seq C2ST {a_seq:.3f} | MMD {mmd_v:.4f}")

## 8 · Summary-feature C2ST: channel decomposition across cohorts

In [ ]:
cohorts = {
    "RecordDiff":       (M_s, Y_s, c_feat),
    "independent-mask": (M_indep, Y_indep, c_feat),
    "mask-replay":      (M_replay, Y_replay, c_feat),
}
rows = {}
for name, (mm, yy, cc) in cohorts.items():
    r = {}
    for ch in ["mask", "value", "joint"]:
        Xr = summary_features(m_real, y_real, c_real, channel=ch)
        Xs = summary_features(mm, yy, cc, channel=ch)
        res = c2st(Xr, Xs, n_splits=C2ST_SPLITS, seed=SEED)
        r[ch] = res
    r["mmd"] = mmd2_rbf(summary_features(m_real, y_real, c_real),
                        summary_features(mm, yy, cc), max_n=MMD_MAX_N)
    rows[name] = r

print(f"{'cohort':18s} | {'mask C2ST':>18s} {'value C2ST':>18s} {'joint C2ST':>18s} | {'MMD':>7s}")
print("-" * 92)
def f(res): return f"{res['acc']:.3f}[{res['acc_lo']:.2f},{res['acc_hi']:.2f}]"
for name, r in rows.items():
    print(f"{name:18s} | {f(r['mask']):>18s} {f(r['value']):>18s} {f(r['joint']):>18s} | {r['mmd']:7.4f}")

obs, pval = c2st_permutation_pvalue(summary_features(m_real, y_real, c_real),
                                    summary_features(M_s, Y_s, c_feat), n_perm=N_PERM, seed=SEED)
print(f"\nRecordDiff joint C2ST = {obs:.3f} | permutation p(> chance) = {pval:.3f} "
      f"({'distinguishable' if pval < 0.05 else 'not distinguishable from chance'})")

## 9 · Sequence C2ST

In [ ]:
mu = np.zeros(V, np.float32); sd = np.ones(V, np.float32)
for v in range(V):
    vals = y_real[:, :, v][m_real[:, :, v] > 0.5]
    if vals.size > 50:
        mu[v] = vals.mean(); sd[v] = max(vals.std(), 1e-3)
I_real = seq_input(m_real, y_real, mu, sd)
seq_rows = {}
for name, (mm, yy, _) in cohorts.items():
    acc, auc = seq_c2st(I_real, seq_input(mm, yy, mu, sd), epochs=SEQ_EPOCHS, hidden=SEQ_HIDDEN, seed=SEED)
    seq_rows[name] = (acc, auc)
    print(f"{name:18s} | seq C2ST acc {acc:.3f} | AUROC {auc:.3f}")
print("\n(≈0.50 = indistinguishable; RecordDiff should sit below independent-mask.)")

## 10 · Distributional diagnostics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
col = {"protocol": "tab:blue", "acuity": "tab:orange", "triggered": "tab:red"}
cols = [col[c] for c in VAR_CLASS]
rd = m_real.mean((0, 1)); sd_ = M_s.mean((0, 1))
ax[0].scatter(rd, sd_, c=cols, s=28)
lim = [0, max(rd.max(), sd_.max()) * 1.05]; ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set_xlabel("real mask density"); ax[0].set_ylabel("synthetic mask density")
ax[0].set_title("per-variable mask density (RecordDiff)")

ksv = ks_per_variable(y_real, m_real, Y_s, M_s)
order = np.argsort(-np.nan_to_num(ksv))
ax[1].bar(range(V), ksv[order], color=[cols[i] for i in order])
ax[1].set_xticks(range(V)); ax[1].set_xticklabels([VAR_NAMES[i] for i in order], rotation=90, fontsize=7)
ax[1].set_ylabel("KS statistic (value dist.)"); ax[1].set_title("per-variable value KS: real vs synthetic")
plt.tight_layout(); plt.show()

print("distributional summary (RecordDiff vs real):")
print(f"  mask-density MAE           : {np.abs(rd - sd_).mean():.4f}")
print(f"  value KS (median / max)    : {np.nanmedian(ksv):.3f} / {np.nanmax(ksv):.3f}")
print(f"  observations-per-record KS : {obs_per_record_ks(m_real, M_s):.3f}")
print(f"  co-occurrence Frobenius    : {frob(cooccurrence(m_real), cooccurrence(M_s)):.3f}  "
      f"(indep-mask: {frob(cooccurrence(m_real), cooccurrence(M_indep)):.3f})")
print(f"  value-correlation Frobenius: {frob(value_corr_nan(y_real, m_real), value_corr_nan(Y_s, M_s)):.3f}")

In [ ]:
import torch.nn as nn
from sklearn.metrics import roc_auc_score

class BiLSTMDisc(nn.Module):
    def __init__(self, in_dim, hidden=48):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.head = nn.Linear(2 * hidden, 1)
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.head(torch.cat([hn[0], hn[1]], dim=-1)).squeeze(-1)

def bilstm_c2st(Ir, Is, epochs=8, hidden=48, batch=256, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    n = min(len(Ir), len(Is)); g = np.random.default_rng(seed)
    Ir = Ir[g.choice(len(Ir), n, replace=False)]; Is = Is[g.choice(len(Is), n, replace=False)]
    X = np.concatenate([Ir, Is], 0); lab = np.r_[np.ones(n), np.zeros(n)].astype(np.float32)
    p = g.permutation(2 * n); X, lab = X[p], lab[p]; ntr = int(1.6 * n)
    Xtr = torch.from_numpy(X[:ntr]).to(device); ytr = torch.from_numpy(lab[:ntr]).to(device)
    Xte = torch.from_numpy(X[ntr:]).to(device); yte = lab[ntr:]
    net = BiLSTMDisc(Ir.shape[-1], hidden).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for _ in range(epochs):
        net.train(); order = torch.randperm(len(Xtr))
        for s in range(0, len(order), batch):
            bi = order[s:s + batch]; opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(net(Xtr[bi]), ytr[bi])
            loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pr = torch.sigmoid(net(Xte)).cpu().numpy()
    acc = ((pr > 0.5).astype(int) == yte.astype(int)).mean()
    auc = roc_auc_score(yte, pr) if len(np.unique(yte)) > 1 else float("nan")
    return float(acc), float(auc)

def val_input(m, y, mu, sd):
    return np.nan_to_num(((y - mu[None, None, :]) / sd[None, None, :]) * m).astype(np.float32)

mu = np.zeros(V, np.float32); sd = np.ones(V, np.float32)
for v in range(V):
    vv = y_real[:, :, v][m_real[:, :, v] > 0.5]
    if vv.size > 50:
        mu[v] = vv.mean(); sd[v] = max(vv.std(), 1e-3)

Ij_real = seq_input(m_real, y_real, mu, sd)
Iv_real = val_input(m_real, y_real, mu, sd)

print(f"{'cohort':18s} | {'JOINT acc':>9s} {'AUROC':>6s} | {'VALUES-ONLY acc':>15s} {'AUROC':>6s}")
print("-" * 74)
for name, (mm, yy, _) in cohorts.items():
    aj, uj = bilstm_c2st(Ij_real, seq_input(mm, yy, mu, sd), epochs=8, seed=0)
    av, uv = bilstm_c2st(Iv_real, val_input(mm, yy, mu, sd), epochs=8, seed=0)
    print(f"{name:18s} | {aj:9.3f} {uj:6.3f} | {av:15.3f} {uv:6.3f}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

def _feat(m, y, ch):
    return summary_features(m, y, None, channel=ch)

def _pca2(Xr, Xs):
    sc = StandardScaler().fit(np.vstack([Xr, Xs]))
    Z = PCA(n_components=2, random_state=0).fit_transform(sc.transform(np.vstack([Xr, Xs])))
    return Z[:len(Xr)], Z[len(Xr):]

def _scatter(ax, Pr, Ps, title):
    ax.scatter(Ps[:, 0], Ps[:, 1], s=6, alpha=0.30, c="tab:red", label="synthetic")
    ax.scatter(Pr[:, 0], Pr[:, 1], s=6, alpha=0.30, c="tab:blue", label="real")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=2.5, framealpha=0.9, loc="upper right")

fig, ax = plt.subplots(2, 2, figsize=(13, 11))

Xr, Xs = _feat(m_real, y_real, "joint"), _feat(M_s, Y_s, "joint")
Pr, Ps = _pca2(Xr, Xs)
_scatter(ax[0, 0], Pr, Ps, "PCA - joint (mask + value) features")

g = np.random.default_rng(0); k = min(1200, len(Xr), len(Xs))
ir, isyn = g.choice(len(Xr), k, replace=False), g.choice(len(Xs), k, replace=False)
pool = StandardScaler().fit_transform(np.vstack([Xr[ir], Xs[isyn]]))
pool = PCA(n_components=min(50, pool.shape[1]), random_state=0).fit_transform(pool)
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(pool)
_scatter(ax[0, 1], Z[:k], Z[k:], f"t-SNE - joint features (n={k}/class)")

Xr, Xs = _feat(m_real, y_real, "mask"), _feat(M_s, Y_s, "mask")
_scatter(ax[1, 0], *_pca2(Xr, Xs), "PCA - mask-only features")

Xr, Xs = _feat(m_real, y_real, "value"), _feat(M_s, Y_s, "value")
_scatter(ax[1, 1], *_pca2(Xr, Xs), "PCA - value-only features")

plt.tight_layout(); plt.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BURG, BLUE = "#8C2F39", "#1F4E79"
SEED, QLO, QHI = 0, 0.005, 0.995

def _feat(m, y, ch):
    return summary_features(m, y, None, channel=ch)

def _pca2(Xr, Xs, seed=SEED):
    """Fit scaler+PCA on the COMBINED data so both clouds share one coordinate system."""
    both = np.vstack([Xr, Xs])
    sc = StandardScaler().fit(both)
    p = PCA(n_components=2, random_state=seed).fit(sc.transform(both))
    Z = p.transform(sc.transform(both))
    return Z[:len(Xr)], Z[len(Xr):], p.explained_variance_ratio_

def _balance(Xr, Xs, seed=SEED):
    """Equal n per group: unequal N makes one cloud look denser/wider for non-distributional reasons."""
    g = np.random.default_rng(seed); k = min(len(Xr), len(Xs))
    return Xr[g.choice(len(Xr), k, replace=False)], Xs[g.choice(len(Xs), k, replace=False)], k

def _panel(ax, Pr, Ps, evr, title, k):
    ax.scatter(Pr[:, 0], Pr[:, 1], s=4, alpha=0.25, c=BLUE, label="real",
               linewidths=0, rasterized=True)
    ax.scatter(Ps[:, 0], Ps[:, 1], s=4, alpha=0.25, c=BURG, label="synthetic",
               linewidths=0, rasterized=True)
    A = np.vstack([Pr, Ps])
    xlo, xhi = np.quantile(A[:, 0], [QLO, QHI]); ylo, yhi = np.quantile(A[:, 1], [QLO, QHI])
    mx, my = 0.06 * (xhi - xlo), 0.06 * (yhi - ylo)
    ax.set_xlim(xlo - mx, xhi + mx); ax.set_ylim(ylo - my, yhi + my)
    ax.set_xlabel(f"PC1 ({100*evr[0]:.1f}% var)", fontsize=9)
    ax.set_ylabel(f"PC2 ({100*evr[1]:.1f}% var)", fontsize=9)
    ax.set_title(title, fontsize=10.5)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_linewidth(0.8); sp.set_color("#5B6472")

fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.3), dpi=200)
stats = {}
for j, (ch, title) in enumerate([("mask", "Measurement channel (mask)"),
                                 ("value", "Value channel")]):
    Xr, Xs, k = _balance(_feat(m_real, y_real, ch), _feat(M_s, Y_s, ch))
    Pr, Ps, evr = _pca2(Xr, Xs)
    _panel(ax[j], Pr, Ps, evr, title, k)
    stats[ch] = (evr, k)

h, l = ax[0].get_legend_handles_labels()
leg = fig.legend(h, l, loc="lower center", ncol=2, frameon=False, fontsize=9,
                 markerscale=3.0, bbox_to_anchor=(0.5, -0.02))
for lh in leg.legend_handles: lh.set_alpha(1.0)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(OUT_PDF, bbox_inches="tight"); plt.savefig(OUT_PNG, bbox_inches="tight")
plt.show()

for ch, (evr, k) in stats.items():
    print(f"{ch:6s}: n={k}/group | PC1 {100*evr[0]:.1f}% + PC2 {100*evr[1]:.1f}% = {100*evr[:2].sum():.1f}% variance")
print(f"\nsaved -> {OUT_PDF}\n         {OUT_PNG}")